# 🏨 Hotel Revenue Analytics
> *Exploring booking trends, revenue drivers, and cancellation patterns across a hotel portfolio.*

**Dataset:** 200 bookings | City Hotels & Resort Hotels | 12 months  
**Tools:** Python · Pandas · Matplotlib · Seaborn  
**Business Question:** *What drives revenue performance — and where are we losing money?*

---

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'sans-serif'

print("✅ Libraries loaded successfully")

In [ ]:
# Load the dataset
df = pd.read_csv('data/hotel_bookings_sample.csv')

# Quick overview
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumn overview:")
df.dtypes

In [ ]:
# Basic stats
print("📊 Key metrics:")
print(f"  Total bookings:      {len(df)}")
print(f"  Cancellation rate:   {df['is_canceled'].mean()*100:.1f}%")
print(f"  Avg daily rate:      €{df['adr'].mean():.2f}")
print(f"  Total revenue:       €{df['total_revenue'].sum():,.2f}")
print(f"  Avg stay length:     {df['stays_in_nights'].mean():.1f} nights")

## 📈 1. Monthly Revenue Trend

In [ ]:
# Monthly revenue
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('arrival_month')['total_revenue'].sum().reindex(month_order)

fig, ax = plt.subplots()
bars = ax.bar(monthly.index, monthly.values, color=sns.color_palette("Blues_d", len(monthly)))
ax.set_title('Total Revenue by Month', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (€)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))

# Annotate peak month
peak = monthly.idxmax()
ax.bar(peak, monthly[peak], color='#1a3a5c', label=f'Peak: {peak}')
ax.legend()
plt.tight_layout()
plt.savefig('monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n💡 Insight: Peak revenue month is {peak} (€{monthly[peak]:,.0f})")

## 🏷️ 2. Hotel Type Comparison

In [ ]:
# City vs Resort
hotel_stats = df.groupby('hotel').agg(
    avg_adr=('adr','mean'),
    total_revenue=('total_revenue','sum'),
    cancellation_rate=('is_canceled','mean'),
    avg_stay=('stays_in_nights','mean')
).round(2)

print("🏨 Hotel Performance Comparison:")
print(hotel_stats.to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['avg_adr', 'total_revenue', 'cancellation_rate']
titles = ['Avg Daily Rate (€)', 'Total Revenue (€)', 'Cancellation Rate']
colors = ['#2196F3', '#4CAF50', '#F44336']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    ax.bar(hotel_stats.index, hotel_stats[metric], color=color, alpha=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('City Hotel vs Resort Hotel — Key Metrics', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('hotel_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 🌍 3. Top Source Markets

In [ ]:
# Revenue by country
top_countries = df.groupby('country')['total_revenue'].sum().sort_values(ascending=True).tail(8)

fig, ax = plt.subplots(figsize=(9,5))
bars = ax.barh(top_countries.index, top_countries.values,
               color=sns.color_palette("viridis", len(top_countries)))
ax.set_title('Revenue by Source Market — Top 8', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Revenue (€)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))

for bar, val in zip(bars, top_countries.values):
    ax.text(val + 100, bar.get_y() + bar.get_height()/2,
            f'€{val:,.0f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('top_markets.png', dpi=150, bbox_inches='tight')
plt.show()

## ⚠️ 4. Cancellation Analysis

In [ ]:
# Cancellations by channel
cancel_by_channel = df.groupby('distribution_channel').agg(
    total=('is_canceled','count'),
    cancelled=('is_canceled','sum')
)
cancel_by_channel['rate'] = (cancel_by_channel['cancelled'] / cancel_by_channel['total'] * 100).round(1)
cancel_by_channel = cancel_by_channel.sort_values('rate', ascending=False)

fig, ax = plt.subplots(figsize=(9,4))
colors = ['#e74c3c' if r > 30 else '#f39c12' if r > 20 else '#2ecc71' for r in cancel_by_channel['rate']]
ax.bar(cancel_by_channel.index, cancel_by_channel['rate'], color=colors)
ax.set_title('Cancellation Rate by Distribution Channel', fontsize=13, fontweight='bold')
ax.set_ylabel('Cancellation Rate (%)')
ax.axhline(y=cancel_by_channel['rate'].mean(), color='navy', linestyle='--', alpha=0.7,
           label=f"Average: {cancel_by_channel['rate'].mean():.1f}%")
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('cancellation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 💼 Key Business Insights

| Finding | Implication |
|---|---|
| Summer months drive peak revenue | Prioritise upsells & longer stays in Jun–Aug |
| Resort hotels charge higher ADR | Consider shifting marketing mix toward leisure |
| High cancellation rate in some channels | Review overbooking policy for Online TA |
| Spain & Germany are top source markets | Focus CRM and loyalty campaigns on these markets |

---
*Analysis by [Your Name] · Data Analyst Portfolio Project*